In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — IMPORTS                                            ║
# ╚══════════════════════════════════════════════════════════════╝
import os
import warnings
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

import timm
from timm.data import resolve_data_config, create_transform

from transformers import AutoTokenizer, AutoModel

import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


Using device: cuda


In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — CONFIG                                             ║
# ╚══════════════════════════════════════════════════════════════╝
TRAIN_CSV       = '/kaggle/input/competitions/petfinder-adoption-prediction/train/train.csv'
TEST_CSV        = '/kaggle/input/competitions/petfinder-adoption-prediction/test/test.csv'
TRAIN_IMAGE_DIR = '/kaggle/input/competitions/petfinder-adoption-prediction/train_images'
TEST_IMAGE_DIR  = '/kaggle/input/competitions/petfinder-adoption-prediction/test_images'

TEXT_COL  = 'Description'
IMG_COL   = 'PetID'
LABEL_COL = 'AdoptionSpeed'

NUM_COLS = ['Age', 'Quantity', 'Fee']
CAT_COLS = [
    'Type', 'Breed1', 'Breed2', 'Gender',
    'Color1', 'Color2', 'Color3',
    'MaturitySize', 'FurLength',
    'Vaccinated', 'Dewormed', 'Sterilized', 'Health',
]

PROJ_DIM  = 256    # toutes les modalités projettent vers cette dim dans la fusion
#BATCH_SIZE = 32
BATCH_SIZE = 8          # ← réduit pour tenir en VRAM
GRAD_ACCUM_STEPS = 4    # ← simule batch_size=32 sans dépasser la mémoire
                        #   (on accumule 4 mini-batches avant de faire optimizer.step)
EPOCHS    = 20

# LR séparés : pretrained → très bas, from-scratch → plus élevé
LR_PRETRAINED = 1e-4
LR_HEAD       = 1e-3

# ── Paramètres Early Stopping du modèle neural ────────────────
# Arrête l'entraînement si le QWK de validation ne s'améliore pas
# pendant PATIENCE epochs consécutives.
PATIENCE        = 5   # nombre d'epochs sans amélioration avant arrêt
VAL_SPLIT       = 0.1 # fraction du train utilisée pour la validation


In [3]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — LOAD RAW DATA + TRAIN/VAL SPLIT                    ║
# ║  Nécessité du split : le QWK affiché en Cell 13 était celui  ║
# ║  du TRAIN SET → métrique trop optimiste, overfit non détecté ║
# ║  On crée un val set FIXE pour surveiller la vraie perf.      ║
# ╚══════════════════════════════════════════════════════════════╝
from sklearn.model_selection import train_test_split

train_raw = pd.read_csv(TRAIN_CSV)
test_raw  = pd.read_csv(TEST_CSV)

print('Train shape:', train_raw.shape)
print('Test  shape:', test_raw.shape)
print('\nLabel distribution:')
print(train_raw[LABEL_COL].value_counts().sort_index())

# Split stratifié : garde la distribution des classes
# dans le train et dans le val set.
train_df, val_df = train_test_split(
    train_raw,
    test_size=VAL_SPLIT,
    random_state=42,
    stratify=train_raw[LABEL_COL]  # stratifié → même distribution label
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'\nTrain : {len(train_df)} samples | Val : {len(val_df)} samples')


Train shape: (14993, 24)
Test  shape: (3972, 23)

Label distribution:
AdoptionSpeed
0     410
1    3090
2    4037
3    3259
4    4197
Name: count, dtype: int64

Train : 13493 samples | Val : 1500 samples


In [4]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — TABULAR PREP                                       ║
# ║  Le scaler est fitté uniquement sur le TRAIN set (pas val).  ║
# ║  Évite la fuite de données (data leakage).                   ║
# ╚══════════════════════════════════════════════════════════════╝
def build_tabular(df, scaler=None, fit=False):
    """Encode les features tabulaires : normalisation + OHE.
    Args:
        df     : DataFrame source
        scaler : StandardScaler pré-fitté (None si fit=True)
        fit    : si True, fitte le scaler sur df (uniquement pour le train)
    Returns:
        num_df : features numériques normalisées
        ohe    : one-hot encoding des catégorielles
        scaler : scaler (fitté ou passé en argument)
    """
    d = df.copy()

    num = d[NUM_COLS].fillna(0).astype(np.float32)
    if fit:
        scaler  = StandardScaler()
        num_arr = scaler.fit_transform(num)   # fit + transform sur train
    else:
        num_arr = scaler.transform(num)       # transform seul sur val/test

    ohe    = pd.get_dummies(d[CAT_COLS].astype(str))
    num_df = pd.DataFrame(num_arr, columns=NUM_COLS, index=d.index)
    return num_df, ohe, scaler


# Fit sur le train, transform sur val et test
num_tr, ohe_tr, scaler = build_tabular(train_df, fit=True)
num_val, ohe_val, _    = build_tabular(val_df,   scaler=scaler)
num_te, ohe_te, _      = build_tabular(test_raw, scaler=scaler)

# Aligne les colonnes OHE : val/test peuvent avoir des catégories absentes
ohe_tr, ohe_val = ohe_tr.align(ohe_val, join='left', axis=1, fill_value=0)
ohe_tr, ohe_te  = ohe_tr.align(ohe_te,  join='left', axis=1, fill_value=0)

tab_train = np.concatenate([num_tr.values,  ohe_tr.values.astype(np.float32)],  axis=1)
tab_val   = np.concatenate([num_val.values, ohe_val.values.astype(np.float32)], axis=1)
tab_test  = np.concatenate([num_te.values,  ohe_te.values.astype(np.float32)],  axis=1)
TAB_DIM   = tab_train.shape[1]

print(f'Tabular dim: {TAB_DIM}')


Tabular dim: 352


In [5]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — TEXT PREP                                          ║
# ╚══════════════════════════════════════════════════════════════╝
# Remplacement des NaN par une chaîne neutre pour ne pas planter
# le tokenizer (NaN n'est pas une str).
txt_train = train_df[TEXT_COL].fillna('no description').astype(str).tolist()
txt_val   = val_df[TEXT_COL].fillna('no description').astype(str).tolist()
txt_test  = test_raw[TEXT_COL].fillna('no description').astype(str).tolist()

print(f'Sample text: {txt_train[0][:80]}')


Sample text: This doggie was found by my uncle said to be somewhere near Mont Kiara. Doesn't 


In [6]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — IMAGE PATHS PREP                                   ║
# ╚══════════════════════════════════════════════════════════════╝
def get_image_path(pet_id, img_dir):
    """Retourne le chemin de la première image d'un pet, None si absente."""
    p = Path(img_dir) / f'{pet_id}-1.jpg'
    return str(p) if p.exists() else None

img_paths_train = [get_image_path(pid, TRAIN_IMAGE_DIR) for pid in train_df[IMG_COL]]
img_paths_val   = [get_image_path(pid, TRAIN_IMAGE_DIR) for pid in val_df[IMG_COL]]
img_paths_test  = [get_image_path(pid, TEST_IMAGE_DIR)  for pid in test_raw[IMG_COL]]

print(f'Missing images — train: {sum(p is None for p in img_paths_train)} '
      f'| val: {sum(p is None for p in img_paths_val)} '
      f'| test: {sum(p is None for p in img_paths_test)}')


Missing images — train: 303 | val: 38 | test: 114


In [7]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — LABELS                                             ║
# ╚══════════════════════════════════════════════════════════════╝
labels_train = train_df[LABEL_COL].values.astype(np.int64)
labels_val   = val_df[LABEL_COL].values.astype(np.int64)
NUM_CLASSES  = len(np.unique(labels_train))

print(f'Number of classes: {NUM_CLASSES}')
print(f'Train labels: {len(labels_train)} | Val labels: {len(labels_val)}')


Number of classes: 5
Train labels: 13493 | Val labels: 1500


In [8]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — IMAGE TRANSFORM                                    ║
# ║  Deux transforms distincts :                                 ║
# ║    • train : augmentation → réduit l'overfit                 ║
# ║    • val/test : déterministe → évaluation reproductible      ║
# ║  Normalisation ImageNet (même stats qu'EfficientNet-B3)      ║
# ╚══════════════════════════════════════════════════════════════╝
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# Train : augmentation légère pour régulariser sans déformer les animaux
train_img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Val / test : pas d'augmentation → résultats déterministes et comparables
val_img_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Image transforms prêts.')
print(f'  Train : Resize(224) + Flip + ColorJitter + Rotation + Normalize')
print(f'  Val   : Resize(224) + Normalize')

Image transforms prêts.
  Train : Resize(224) + Flip + ColorJitter + Rotation + Normalize
  Val   : Resize(224) + Normalize


In [9]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — DATASET & DATALOADER                               ║
# ║  PetDataset gère les 3 modalités en parallèle.               ║
# ║  collate_fn garde le texte en liste (requis par HuggingFace).║
# ╚══════════════════════════════════════════════════════════════╝

class PetDataset(Dataset):
    def __init__(self, tab, texts, img_paths, labels=None, img_transform=None):
        self.tab           = torch.tensor(tab, dtype=torch.float32)
        self.texts         = texts
        self.img_paths     = img_paths
        self.labels        = labels
        self.img_transform = img_transform
        self.blank_img     = Image.new('RGB', (224, 224), color=0)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tab  = self.tab[idx]
        text = self.texts[idx]
        path = self.img_paths[idx]

        try:
            img = Image.open(path).convert('RGB') if path else self.blank_img
        except Exception:
            img = self.blank_img  # image corrompue → image noire

        if self.img_transform:
            img = self.img_transform(img)

        if self.labels is not None:
            return tab, text, img, self.labels[idx]
        return tab, text, img


def collate_fn(batch):
    if len(batch[0]) == 4:
        tabs, texts, imgs, labels = zip(*batch)
        return (torch.stack(tabs),
                list(texts),
                torch.stack(imgs),
                torch.tensor(labels, dtype=torch.long))
    else:
        tabs, texts, imgs = zip(*batch)
        return torch.stack(tabs), list(texts), torch.stack(imgs)


train_dataset = PetDataset(tab_train, txt_train, img_paths_train,
                            labels=labels_train, img_transform=train_img_transform)
val_dataset   = PetDataset(tab_val,   txt_val,   img_paths_val,
                            labels=labels_val,   img_transform=val_img_transform)
test_dataset  = PetDataset(tab_test,  txt_test,  img_paths_test,
                            labels=None,         img_transform=val_img_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2, collate_fn=collate_fn,
                          pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, collate_fn=collate_fn,
                          pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, collate_fn=collate_fn,
                          pin_memory=True)

# ── Sanity check ──────────────────────────────────────────────
tab_b, txt_b, img_b, lbl_b = next(iter(train_loader))
print(f'Tabular : {tab_b.shape}')
print(f'Text    : {len(txt_b)} strings')
print(f'Images  : {img_b.shape}')
print(f'Labels  : {lbl_b.shape}')
print(f'Train   : {len(train_dataset)} samples | Val : {len(val_dataset)} samples')

Tabular : torch.Size([8, 352])
Text    : 8 strings
Images  : torch.Size([8, 3, 224, 224])
Labels  : torch.Size([8])
Train   : 13493 samples | Val : 1500 samples


In [10]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — MODEL DEFINITIONS                                 ║
# ║  Architecture : tête ORDINALE + OrdinalLoss                  ║
# ║  Pourquoi ordinal et pas CrossEntropy ?                      ║
# ║  AdoptionSpeed 0→4 est ORDONNÉ : se tromper de 3 classes est ║
# ║  pire que se tromper d'1 classe. CrossEntropy traite toutes  ║
# ║  les erreurs comme équivalentes, ce qui nuit au QWK.         ║
# ║  L'ordinal head prédit K-1 seuils binaires : P(y>0)…P(y>3)  ║
# ║  ce qui encode l'ordre directement dans la loss.             ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Encodeur tabulaire ────────────────────────────────────────
# Réseau MLP avec BN + Dropout pour régulariser les features tabulaires.
class TabularEncoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(input_dim),
            nn.Linear(input_dim, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 256),       nn.ReLU(), nn.Dropout(0.2),
        )
    def forward(self, x):
        return self.net(x)


# ── Fusion Classifier avec tête ordinale ─────────────────────
class OrdinalFusionClassifier(nn.Module):
    def __init__(self, img_dim, txt_dim, tab_dim, num_classes, proj_dim=256):
        super().__init__()

        # Projection de chaque modalité vers proj_dim
        # pour éviter qu'une modalité domine par sa taille (img=1536 >> tab=256)
        self.img_proj = nn.Linear(img_dim, proj_dim)
        self.txt_proj = nn.Linear(txt_dim, proj_dim)
        self.tab_proj = nn.Linear(tab_dim, proj_dim)

        # Gates appris : poids dynamiques par sample pour chaque modalité
        # [img_w, txt_w, tab_w] avec Softmax → somme = 1
        self.gate = nn.Sequential(
            nn.Linear(proj_dim * 3, 3),
            nn.Softmax(dim=-1)
        )

        # Bloc de features commun avec normalisation et GELU
        self.feature = nn.Sequential(
            nn.LayerNorm(proj_dim),
            nn.Linear(proj_dim, 128), nn.GELU(), nn.Dropout(0.3),
        )

        # Tête ordinale : num_classes-1 seuils binaires indépendants
        # Exemple pour num_classes=5 :
        #   logits[0] → P(y>0), logits[1] → P(y>1), ..., logits[3] → P(y>3)
        self.ordinal_head = nn.Linear(128, num_classes - 1)

    def forward(self, img_emb, txt_emb, tab_emb):
        # Projection vers dim commune
        img = self.img_proj(img_emb)
        txt = self.txt_proj(txt_emb)
        tab = self.tab_proj(tab_emb)

        # Gates dynamiques par sample [B, 3]
        gates = self.gate(torch.cat([img, txt, tab], dim=-1))

        # Fusion pondérée par les gates
        fused = gates[:, 0:1]*img + gates[:, 1:2]*txt + gates[:, 2:3]*tab

        # Extraction de features → logits ordinaux [B, num_classes-1]
        feat   = self.feature(fused)
        logits = self.ordinal_head(feat)
        return logits, gates


# ── Ordinal Loss ──────────────────────────────────────────────
# Convertit les labels en cibles binaires puis applique BCE.
# Exemple : label=2, num_classes=5 → cibles = [1, 1, 0, 0]
#           signifie P(y>0)=1, P(y>1)=1, P(y>2)=0, P(y>3)=0
def ordinal_loss(logits, labels, num_classes=5):
    targets = torch.zeros(len(labels), num_classes - 1, device=labels.device)
    for i, label in enumerate(labels):
        targets[i, :label] = 1.0
    return F.binary_cross_entropy_with_logits(logits, targets)


# ── Ordinal Prediction ────────────────────────────────────────
# Reconvertit les logits en classe prédite.
# Exemple : sigmoid([2.3, 1.5, -0.2, -1.0]) → [0.9, 0.8, 0.45, 0.27]
#           seuils > 0.5 : [1, 1, 0, 0] → classe = 2
def ordinal_predict(logits):
    probs = torch.sigmoid(logits)       # [B, num_classes-1]
    return (probs > 0.5).sum(dim=-1)    # compte les seuils dépassés → classe


In [11]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — INSTANTIATION DES MODÈLES                         ║
# ╚══════════════════════════════════════════════════════════════╝

# ── EfficientNet-B3 ───────────────────────────────────────────
model_img = timm.create_model('efficientnet_b3', pretrained=True, num_classes=0)
model_img = model_img.float().to(device)
IMG_DIM   = model_img.num_features   # 1536

# ── DeBERTa-v3-small ──────────────────────────────────────────
TOKENIZER = AutoTokenizer.from_pretrained('microsoft/deberta-v3-small')
model_txt = AutoModel.from_pretrained(
    'microsoft/deberta-v3-small',
    torch_dtype=torch.float32
).to(device)

# Gradient checkpointing : -60% VRAM, +20% temps
model_txt.gradient_checkpointing_enable()

# Tous les paramètres libres — le modèle décide lui-même
# quelles features sont utiles via les gates
for param in model_txt.parameters():
    param.requires_grad = True

total = sum(p.numel() for p in model_txt.parameters())
print(f'DeBERTa : {total/1e6:.1f}M params — tous libres')

TXT_DIM = model_txt.config.hidden_size  # 768

# ── TabularEncoder ────────────────────────────────────────────
tab_encoder = TabularEncoder(TAB_DIM).float().to(device)
TAB_ENC_DIM = 256

# ── Fusion Classifier ─────────────────────────────────────────
fusion = OrdinalFusionClassifier(
    img_dim     = IMG_DIM,
    txt_dim     = TXT_DIM,
    tab_dim     = TAB_ENC_DIM,
    num_classes = NUM_CLASSES,
    proj_dim    = PROJ_DIM
).float().to(device)

print(f'IMG_DIM  : {IMG_DIM}')
print(f'TXT_DIM  : {TXT_DIM}')
print(f'TAB_DIM  : {TAB_DIM} → encoder out: {TAB_ENC_DIM}')
print(f'Classes  : {NUM_CLASSES}')

model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DeBERTa : 141.3M params — tous libres
IMG_DIM  : 1536
TXT_DIM  : 768
TAB_DIM  : 352 → encoder out: 256
Classes  : 5


In [12]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — OPTIMIZER & SCHEDULER                             ║
# ║  LR différenciés par encoder :                               ║
# ║    • Pretrained (img, txt) : bas → fine-tune doux            ║
# ║    • From scratch (tab, fusion) : plus élevé → apprentissage ║
# ║  OneCycleLR : warmup 20% + cosine decay                      ║
# ║  → step PAR BATCH ACCUMULÉ (voir Cell 13)                    ║
# ╚══════════════════════════════════════════════════════════════╝
from torch.optim.lr_scheduler import CosineAnnealingLR

optimizer = torch.optim.AdamW([
    {'params': model_img.parameters(),   'lr': 3e-5},
    {'params': model_txt.parameters(),   'lr': 1e-5},
    {'params': tab_encoder.parameters(), 'lr': 1e-4},
    {'params': fusion.parameters(),      'lr': 1e-4},
], weight_decay=1e-2)

scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)

print(f'Optimizer prêt — {EPOCHS} epochs, CosineAnnealingLR')

Optimizer prêt — 20 epochs, CosineAnnealingLR


In [13]:
import copy
from torch.cuda.amp import GradScaler

scaler_amp = GradScaler()


def get_txt_embedding(texts, micro_batch=4):
    all_embs = []
    for i in range(0, len(texts), micro_batch):
        chunk = texts[i:i + micro_batch]
        enc = TOKENIZER(chunk, truncation=True, padding=True,
                        max_length=128, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.cuda.amp.autocast(enabled=False):
            out = model_txt(**enc)
        all_embs.append(out.last_hidden_state[:, 0, :].float())
    return torch.cat(all_embs, dim=0)


@torch.no_grad()
def evaluate(loader):
    model_img.eval(); model_txt.eval()
    tab_encoder.eval(); fusion.eval()

    all_preds, all_labels = [], []
    total_loss, total_n   = 0.0, 0

    for tab_b, txt_b, img_b, lbl_b in loader:
        tab_b = tab_b.float().to(device)
        img_b = img_b.float().to(device)
        lbl_b = lbl_b.to(device)

        img_emb = model_img(img_b).float()
        txt_emb = get_txt_embedding(txt_b)
        tab_emb = tab_encoder(tab_b)

        logits, _ = fusion(img_emb, txt_emb, tab_emb)
        logits = logits.float()
        loss   = ordinal_loss(logits, lbl_b, num_classes=NUM_CLASSES)

        if torch.isnan(loss):
            continue

        preds = ordinal_predict(logits)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(lbl_b.cpu().numpy())
        total_loss += loss.item() * len(lbl_b)
        total_n    += len(lbl_b)

    if total_n == 0:
        return 0.0, float('nan')

    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    return qwk, total_loss / total_n


# ── Early stopping state ──────────────────────────────────────
best_val_qwk     = -1.0
patience_counter = 0
best_model_state = None

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train QWK':>9} | "
      f"{'Val Loss':>8} | {'Val QWK':>7} | Gates (img/txt/tab)")
print('-' * 85)

for epoch in range(EPOCHS):

    model_img.train(); model_txt.train()
    tab_encoder.train(); fusion.train()

    total_loss, total_samples = 0.0, 0
    nan_batches = 0
    train_preds, train_labels = [], []
    gates = None

    optimizer.zero_grad()

    for step, (tab_b, txt_b, img_b, lbl_b) in enumerate(train_loader):
        tab_b = tab_b.float().to(device)
        img_b = img_b.float().to(device)
        lbl_b = lbl_b.to(device)

        with torch.cuda.amp.autocast():
            img_emb = model_img(img_b).float()
        txt_emb = get_txt_embedding(txt_b)
        tab_emb = tab_encoder(tab_b)

        if any(torch.isnan(e).any() for e in [img_emb, txt_emb, tab_emb]):
            nan_batches += 1
            continue

        logits, gates = fusion(img_emb, txt_emb, tab_emb)
        loss = ordinal_loss(logits, lbl_b, num_classes=NUM_CLASSES) / GRAD_ACCUM_STEPS

        if torch.isnan(loss):
            nan_batches += 1
            continue

        scaler_amp.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler_amp.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                list(model_img.parameters()) + list(model_txt.parameters()) +
                list(tab_encoder.parameters()) + list(fusion.parameters()),
                max_norm=1.0
            )
            scaler_amp.step(optimizer)
            scaler_amp.update()
            optimizer.zero_grad()

        preds = ordinal_predict(logits)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(lbl_b.cpu().numpy())
        total_loss    += loss.item() * GRAD_ACCUM_STEPS * len(lbl_b)
        total_samples += len(lbl_b)

    # CosineAnnealingLR : step par epoch
    scheduler.step()

    if total_samples == 0:
        print(f'Epoch {epoch+1:02d} — tous les batches NaN.')
        continue

    train_loss = total_loss / total_samples
    train_qwk  = cohen_kappa_score(train_labels, train_preds, weights='quadratic')

    torch.cuda.empty_cache()
    val_qwk, val_loss = evaluate(val_loader)
    torch.cuda.empty_cache()

    gates_str = 'gates=N/A'
    if gates is not None:
        g = gates.detach().float().mean(dim=0)
        gates_str = f'img={g[0]:.2f} txt={g[1]:.2f} tab={g[2]:.2f}'

    print(f'{epoch+1:>5d} | {train_loss:>10.4f} | {train_qwk:>9.4f} | '
          f'{val_loss:>8.4f} | {val_qwk:>7.4f} | {gates_str}'
          + (f' | NaN:{nan_batches}' if nan_batches else ''))

    if val_qwk > best_val_qwk:
        best_val_qwk     = val_qwk
        patience_counter = 0
        best_model_state = {
            'fusion':      copy.deepcopy(fusion.state_dict()),
            'tab_encoder': copy.deepcopy(tab_encoder.state_dict()),
            'model_img':   copy.deepcopy(model_img.state_dict()),
            'model_txt':   copy.deepcopy(model_txt.state_dict()),
        }
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\n⚡ Early stopping epoch {epoch+1} '
                  f'(patience={PATIENCE}, best val QWK={best_val_qwk:.4f})')
            break

if best_model_state is not None:
    fusion.load_state_dict(best_model_state['fusion'])
    tab_encoder.load_state_dict(best_model_state['tab_encoder'])
    model_img.load_state_dict(best_model_state['model_img'])
    model_txt.load_state_dict(best_model_state['model_txt'])
    print(f'\n✅ Meilleurs poids rechargés (val QWK = {best_val_qwk:.4f})')

Epoch | Train Loss | Train QWK | Val Loss | Val QWK | Gates (img/txt/tab)
-------------------------------------------------------------------------------------
    1 |     0.4733 |    0.1884 |   0.4519 |  0.2930 | img=0.16 txt=0.04 tab=0.80
    2 |     0.4401 |    0.3486 |   0.4601 |  0.2682 | img=0.15 txt=0.02 tab=0.84
    3 |     0.4175 |    0.4337 |   0.4859 |  0.2866 | img=0.21 txt=0.04 tab=0.74
    4 |     0.3931 |    0.5159 |   0.4745 |  0.2834 | img=0.23 txt=0.08 tab=0.69
    5 |     0.3629 |    0.5929 |   0.5006 |  0.2995 | img=0.24 txt=0.06 tab=0.70
    6 |     0.3379 |    0.6460 |   0.5484 |  0.2746 | img=0.22 txt=0.08 tab=0.71
    7 |     0.3101 |    0.6988 |   0.5391 |  0.2942 | img=0.19 txt=0.09 tab=0.71
    8 |     0.2865 |    0.7412 |   0.5857 |  0.2867 | img=0.35 txt=0.09 tab=0.56
    9 |     0.2661 |    0.7727 |   0.6026 |  0.2880 | img=0.30 txt=0.08 tab=0.62
   10 |     0.2431 |    0.8052 |   0.6478 |  0.3010 | img=0.27 txt=0.09 tab=0.64
   11 |     0.2282 |    0.8225

In [14]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 14 — POURQUOI LGBM ? + CROSS-VALIDATION + EARLY STOP  ║
# ║                                                              ║
# ║  Pourquoi tester LGBM en plus du réseau de neurones ?        ║
# ║  ─────────────────────────────────────────────────────────── ║
# ║  1. COMPLÉMENTARITÉ : le neural net apprend des patterns     ║
# ║     non-linéaires dans l'image et le texte, mais les arbres  ║
# ║     gradient boosting sont très efficaces sur les features   ║
# ║     tabulaires structurées (âge, race, prix...).             ║
# ║  2. ROBUSTESSE : LGBM overfite moins facilement sur les      ║
# ║     features continues grâce à l'histogramme.                ║
# ║  3. ENSEMBLE : combiner les deux modèles donne généralement  ║
# ║     un QWK supérieur aux deux pris séparément (diversité     ║
# ║     des erreurs → annulation partielle).                     ║
# ║  4. VALIDATION FIABLE : LGBM avec StratifiedKFold + OOF     ║
# ║     donne une estimation non-biaisée du QWK final, alors     ║
# ║     que le neural net n'a qu'un seul val split.              ║
# ╚══════════════════════════════════════════════════════════════╝

# ── 1. Extraction des embeddings ─────────────────────────────
# On utilise le meilleur checkpoint chargé en fin de Cell 13.
@torch.no_grad()
def extract_embeddings(loader):
    """Extrait et concatène les embeddings des 3 modalités.
    Les modèles sont mis en eval() pour désactiver Dropout/BN train mode.
    """
    model_img.eval(); model_txt.eval(); tab_encoder.eval()
    all_img, all_txt, all_tab, all_labels = [], [], [], []

    for batch in loader:
        if len(batch) == 4:
            tab_b, txt_b, img_b, lbl_b = batch
            all_labels.extend(lbl_b.numpy())
        else:
            tab_b, txt_b, img_b = batch

        tab_b = tab_b.float().to(device)
        img_b = img_b.float().to(device)

        all_img.append(model_img(img_b).float().cpu().numpy())
        all_txt.append(get_txt_embedding(txt_b).cpu().numpy())
        all_tab.append(tab_encoder(tab_b).cpu().numpy())

    # Concaténation des 3 modalités en un seul vecteur [N, 1536+768+256]
    embeddings = np.concatenate([
        np.concatenate(all_img),    # [N, 1536]
        np.concatenate(all_txt),    # [N, 768]
        np.concatenate(all_tab),    # [N, 256]
    ], axis=1)                      # [N, 2560]

    return embeddings, np.array(all_labels) if all_labels else None


# On ré-utilise le train_loader COMPLET (train_df + val_df combinés)
# pour maximiser les données disponibles pour le LGBM après que le neural
# net a déjà été sélectionné. Le val set a servi à choisir le checkpoint.
print('Extraction des embeddings (train complet)...')
# Créer un loader temporaire sur tout le train_raw pour LGBM
full_tab = np.concatenate([tab_train, tab_val], axis=0)
full_txt = txt_train + txt_val
full_img = img_paths_train + img_paths_val
full_lbl = np.concatenate([labels_train, labels_val], axis=0)

full_dataset = PetDataset(full_tab, full_txt, full_img,
                           labels=full_lbl, img_transform=val_img_transform)
full_loader  = DataLoader(full_dataset, batch_size=BATCH_SIZE,
                           shuffle=False, num_workers=2, collate_fn=collate_fn)

X_train_emb, y_train_emb = extract_embeddings(full_loader)

print('Extraction des embeddings test...')
X_test_emb, _ = extract_embeddings(test_loader)

print(f'Shape embeddings train : {X_train_emb.shape}')
print(f'Shape embeddings test  : {X_test_emb.shape}')


# ── 2. Paramètres LGBM (régularisation renforcée pour éviter l'overfit)
# num_leaves réduit, min_child_samples élevé, subsample/colsample bas,
# reg_alpha/reg_lambda élevés = arbres moins complexes, meilleure généralisation.
LGB_PARAMS = dict(
    n_estimators      = 1000,        # beaucoup d'arbres, early stopping coupe
    learning_rate     = 0.02,        # LR bas = meilleure généralisation
    num_leaves        = 20,          # arbres moins larges
    max_depth         = 6,           # arbres moins profonds
    min_child_samples = 100,         # feuilles plus denses
    subsample         = 0.7,         # subsampling des lignes
    subsample_freq    = 1,
    colsample_bytree  = 0.5,         # subsampling des colonnes
    reg_alpha         = 2.0,         # régularisation L1
    reg_lambda        = 5.0,         # régularisation L2
    objective         = 'multiclass',
    num_class         = NUM_CLASSES,
    random_state      = 42,
    verbose           = -1,
)


# ── 3. Cross-validation StratifiedKFold (5 folds) ────────────
# OOF = Out-Of-Fold : chaque sample est prédit exactement une fois
# en tant que validation → estimation non-biaisée du QWK réel.
N_FOLDS     = 5
skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_preds   = np.zeros(len(X_train_emb), dtype=int)
test_probas = np.zeros((len(X_test_emb), NUM_CLASSES))  # probas moyennées

print(f'\n── Cross-validation {N_FOLDS} folds ──────────────────────────')
fold_qwks = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_emb, y_train_emb)):
    X_tr, X_val = X_train_emb[train_idx], X_train_emb[val_idx]
    y_tr, y_val = y_train_emb[train_idx], y_train_emb[val_idx]

    lgb_fold = lgb.LGBMClassifier(**LGB_PARAMS)

    # Early stopping sur le val set de ce fold
    # stopping_rounds=50 : arrête si pas d'amélioration pendant 50 rounds
    lgb_fold.fit(
        X_tr, y_tr,
        eval_set    = [(X_val, y_val)],
        eval_metric = 'multi_logloss',
        callbacks   = [
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=200),
        ]
    )

    # Prédictions OOF sur le fold val
    val_preds           = lgb_fold.predict(X_val)
    oof_preds[val_idx]  = val_preds

    # QWK de validation du fold (estimation honnête)
    fold_qwk = cohen_kappa_score(y_val, val_preds, weights='quadratic')
    fold_qwks.append(fold_qwk)
    print(f'  Fold {fold+1}/{N_FOLDS} | Val QWK : {fold_qwk:.4f} '
          f'| Best iter : {lgb_fold.best_iteration_}')

    # Accumule les probas test → moyenne sur les N folds pour plus de stabilité
    test_probas += lgb_fold.predict_proba(X_test_emb) / N_FOLDS


# ── 4. Métriques OOF finales ──────────────────────────────────
oof_qwk = cohen_kappa_score(y_train_emb, oof_preds, weights='quadratic')
print(f'\n{"="*55}')
print(f'  QWK OOF global      : {oof_qwk:.4f}  ← estimation réelle (non biaisée)')
print(f'  QWK moyen par fold  : {np.mean(fold_qwks):.4f} ± {np.std(fold_qwks):.4f}')
print(f'{"="*55}')


# ── 5. Prédictions finales : ensemble LGBM + Neural ──────────
# Combiner les deux modèles améliore le QWK car ils font
# des erreurs différentes (diversité des modèles).

# LGBM : argmax des probas moyennées sur les 5 folds
lgb_final_preds = np.argmax(test_probas, axis=1)

# Neural : prédictions ordinales sur le test set
neural_preds = []
model_img.eval(); model_txt.eval(); tab_encoder.eval(); fusion.eval()
with torch.no_grad():
    for tab_b, txt_b, img_b in test_loader:
        tab_b = tab_b.float().to(device)
        img_b = img_b.float().to(device)
        img_emb = model_img(img_b).float()
        txt_emb = get_txt_embedding(txt_b)
        tab_emb = tab_encoder(tab_b)
        logits, _ = fusion(img_emb, txt_emb, tab_emb)
        neural_preds.extend(ordinal_predict(logits).cpu().numpy())

neural_preds = np.array(neural_preds)

# Ensemble 50/50 — arrondi et clip pour rester dans [0, NUM_CLASSES-1]
final_preds = np.round(0.5 * lgb_final_preds + 0.5 * neural_preds).astype(int)
final_preds = np.clip(final_preds, 0, NUM_CLASSES - 1)

print(f'\nDistribution des prédictions finales :')
print(pd.Series(final_preds).value_counts().sort_index())


Extraction des embeddings (train complet)...
Extraction des embeddings test...
Shape embeddings train : (14993, 2560)
Shape embeddings test  : (3972, 2560)

── Cross-validation 5 folds ──────────────────────────
[200]	valid_0's multi_logloss: 0.896456
[400]	valid_0's multi_logloss: 0.830385
[600]	valid_0's multi_logloss: 0.80867
[800]	valid_0's multi_logloss: 0.798977
[1000]	valid_0's multi_logloss: 0.794325
  Fold 1/5 | Val QWK : 0.8130 | Best iter : 999
[200]	valid_0's multi_logloss: 0.881223
[400]	valid_0's multi_logloss: 0.813062
[600]	valid_0's multi_logloss: 0.79011
[800]	valid_0's multi_logloss: 0.778996
[1000]	valid_0's multi_logloss: 0.77409
  Fold 2/5 | Val QWK : 0.8280 | Best iter : 998
[200]	valid_0's multi_logloss: 0.905592
[400]	valid_0's multi_logloss: 0.83916
[600]	valid_0's multi_logloss: 0.8162
[800]	valid_0's multi_logloss: 0.805693
[1000]	valid_0's multi_logloss: 0.800315
  Fold 3/5 | Val QWK : 0.8143 | Best iter : 1000
[200]	valid_0's multi_logloss: 0.893792
[400]	

In [15]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 15 — SAUVEGARDE MODÈLES (.pkl) + SUBMISSION           ║
# ╚══════════════════════════════════════════════════════════════╝
import pickle
import os

SAVE_DIR = '/kaggle/working'
os.makedirs(SAVE_DIR, exist_ok=True)

# ── 1. LightGBM final (dernier fold — pour serving simple) ────
# Note : pour une production sérieuse, sauvegarder tous les folds
# et faire la moyenne de leurs probas comme dans la CV ci-dessus.
lgb_path = os.path.join(SAVE_DIR, 'lgbm_final.pkl')
with open(lgb_path, 'wb') as f:
    pickle.dump(lgb_fold, f)
print(f'✅ LightGBM sauvegardé → {lgb_path}  ({os.path.getsize(lgb_path)/1e6:.1f} MB)')

# ── 2. Scaler tabulaire ───────────────────────────────────────
scaler_path = os.path.join(SAVE_DIR, 'tab_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f'✅ Scaler sauvegardé   → {scaler_path}')

# ── 3. Colonnes OHE ───────────────────────────────────────────
ohe_cols_path = os.path.join(SAVE_DIR, 'ohe_columns.pkl')
with open(ohe_cols_path, 'wb') as f:
    pickle.dump(list(ohe_tr.columns), f)
print(f'✅ Colonnes OHE        → {ohe_cols_path}')

# ── 4. Bundle neural (state_dicts du meilleur checkpoint) ─────
"""
neural_bundle = {
    'fusion_state':      fusion.state_dict(),
    'tab_encoder_state': tab_encoder.state_dict(),
    'img_encoder_state': model_img.state_dict(),
    'txt_encoder_state': model_txt.state_dict(),
    'best_val_qwk':      best_val_qwk,     # QWK val du meilleur checkpoint
}
neural_pkl_path = os.path.join(SAVE_DIR, 'neural_bundle.pkl')
with open(neural_pkl_path, 'wb') as f:
    pickle.dump(neural_bundle, f)
print(f'✅ Neural bundle        → {neural_pkl_path}  ({os.path.getsize(neural_pkl_path)/1e6:.1f} MB)')
"""

# ── 5. Submission CSV ─────────────────────────────────────────
submission = pd.DataFrame({
    'PetID':         test_raw['PetID'],
    'AdoptionSpeed': final_preds,
})
sub_path = os.path.join(SAVE_DIR, 'submission.csv')
submission.to_csv(sub_path, index=False)
print(f'\n✅ Submission → {sub_path}')
print(submission['AdoptionSpeed'].value_counts().sort_index())

# ── 6. Récapitulatif ──────────────────────────────────────────
print('\n── Fichiers dans /kaggle/working ──────────────────────')
for fname in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, fname)
    print(f'  {fname:<35}  {os.path.getsize(fpath)/1e6:>8.2f} MB')


✅ LightGBM sauvegardé → /kaggle/working/lgbm_final.pkl  (11.4 MB)
✅ Scaler sauvegardé   → /kaggle/working/tab_scaler.pkl
✅ Colonnes OHE        → /kaggle/working/ohe_columns.pkl

✅ Submission → /kaggle/working/submission.csv
AdoptionSpeed
0      61
1     453
2    1992
3     568
4     898
Name: count, dtype: int64

── Fichiers dans /kaggle/working ──────────────────────
  .virtual_documents                       0.00 MB
  lgbm_final.pkl                          11.38 MB
  ohe_columns.pkl                          0.00 MB
  submission.csv                           0.05 MB
  tab_scaler.pkl                           0.00 MB


In [16]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 16 — EXPORT DU MEILLEUR FUSION CLASSIFIER              ║
# ║  On sauvegarde TOUT ce qu'il faut pour recharger et          ║
# ║  utiliser le modèle hors Kaggle (refuge, API, chatbot) :     ║
# ║    • fusion_state_dict   : poids du OrdinalFusionClassifier  ║
# ║    • tab_encoder_state   : poids du TabularEncoder           ║
# ║    • img_encoder_state   : poids EfficientNet-B3             ║
# ║    • txt_encoder_state   : poids DeBERTa-v3-small            ║
# ║    • architecture_config : dimensions, num_classes, proj_dim ║
# ║    • scaler + ohe_cols   : préprocessing tabulaire           ║
# ║    • best_val_qwk        : score de référence                ║
# ╚══════════════════════════════════════════════════════════════╝
import pickle, os, torch

SAVE_DIR = '/kaggle/working'

# ── Paquet complet pour déploiement ──────────────────────────
# Ce fichier suffit à lui seul pour faire tourner le modèle
# hors Kaggle (refuge, script batch, API Flask, chatbot).
deployment_bundle = {
    # ── Poids des 4 sous-modèles ─────────────────────────────
    'fusion_state_dict':    fusion.state_dict(),
    'tab_encoder_state':    tab_encoder.state_dict(),
    'img_encoder_state':    model_img.state_dict(),
    'txt_encoder_state':    model_txt.state_dict(),

    # ── Config d'architecture (nécessaire pour réinstancier) ─
    # Sans ces valeurs, impossible de recréer les objets Python
    # avant de charger les state_dict.
    'architecture': {
        'img_dim':     IMG_DIM,       # 1536
        'txt_dim':     TXT_DIM,       # 768
        'tab_enc_dim': TAB_ENC_DIM,   # 256
        'proj_dim':    PROJ_DIM,      # 256
        'num_classes': NUM_CLASSES,   # 5
        'tab_input_dim': TAB_DIM,     # dimension raw des features tabulaires
    },

    # ── Préprocessing tabulaire ───────────────────────────────
    'scaler':   scaler,               # StandardScaler fitté sur le train
    'ohe_cols': list(ohe_tr.columns), # colonnes OHE de référence

    # ── Colonnes utilisées ────────────────────────────────────
    'num_cols': NUM_COLS,
    'cat_cols': CAT_COLS,

    # ── Score de référence ────────────────────────────────────
    'best_val_qwk': best_val_qwk,
}

bundle_path = os.path.join(SAVE_DIR, 'fusion_best_model.pkl')
with open(bundle_path, 'wb') as f:
    pickle.dump(deployment_bundle, f)

size_mb = os.path.getsize(bundle_path) / 1e6
print(f'✅ Fusion best model exporté → {bundle_path}  ({size_mb:.1f} MB)')
print(f'   Val QWK du checkpoint : {best_val_qwk:.4f}')
print(f'   Architecture sauvegardée : {deployment_bundle["architecture"]}')

✅ Fusion best model exporté → /kaggle/working/fusion_best_model.pkl  (612.0 MB)
   Val QWK du checkpoint : 0.3137
   Architecture sauvegardée : {'img_dim': 1536, 'txt_dim': 768, 'tab_enc_dim': 256, 'proj_dim': 256, 'num_classes': 5, 'tab_input_dim': 352}


# 🐾 Guide d'exploitation du Fusion Classifier — Refuge animalier

Le fichier `fusion_best_model.pkl` exporté en Cell 16 contient **tout** ce dont
on a besoin pour prédire l'`AdoptionSpeed` d'un animal sans relancer l'entraînement.

---

## 🔧 Chargement du modèle (commun aux deux modes)

Coller ce bloc en haut de n'importe quel script ou notebook d'exploitation :

```python

In [17]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 18 — CROSS-MODAL TRANSFORMER (CMT)                     ║
# ║                                                              ║
# ║  Différence vs OrdinalFusionClassifier :                     ║
# ║  • Fusion simple : proj(img) + proj(txt) + proj(tab)         ║
# ║    → les modalités ne se voient pas avant la décision        ║
# ║  • CMT : [CLS, img, txt, tab] passent dans un Transformer    ║
# ║    → attention croisée entre modalités avant classification   ║
# ║    → img peut "regarder" le texte, tab peut "regarder" img   ║
# ╚══════════════════════════════════════════════════════════════╝
import torch.nn as nn
import torch

class CrossModalTransformer(nn.Module):
    def __init__(self, img_dim, txt_dim, tab_dim,
                 num_classes, proj_dim=128, nhead=4, num_layers=2, dropout=0.15):
        super().__init__()

        # ── Projection de chaque modalité vers proj_dim ───────
        # BN sur le tabulaire car ses features sont hétérogènes (âge, prix...)
        # LayerNorm sur img/txt car ce sont déjà des représentations denses
        self.img_proj = nn.Sequential(
            nn.Linear(img_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.txt_proj = nn.Sequential(
            nn.Linear(txt_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.tab_proj = nn.Sequential(
            nn.BatchNorm1d(tab_dim),
            nn.Linear(tab_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # ── Token CLS appris ──────────────────────────────────
        # Vecteur de taille [1, 1, proj_dim] optimisé pendant l'entraînement.
        # Son rôle : agréger l'information de img+txt+tab via l'attention.
        # En sortie du Transformer, c'est ce token qu'on classifie.
        self.cls_token = nn.Parameter(torch.zeros(1, 1, proj_dim))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # ── Encodage positionnel par modalité ─────────────────
        # Appris (pas sinusoïdal) : 4 positions → [CLS, img, txt, tab]
        self.pos_embed = nn.Parameter(torch.zeros(1, 4, proj_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        # ── Transformer Encoder ───────────────────────────────
        # Pre-LN (norm_first=True) : plus stable que Post-LN en fine-tuning
        # batch_first=True : convention [B, seq_len, dim]
        encoder_layer = nn.TransformerEncoderLayer(
            d_model         = proj_dim,
            nhead           = nhead,
            dim_feedforward = proj_dim * 4,
            dropout         = dropout,
            batch_first     = True,
            norm_first      = True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers = num_layers,
            norm       = nn.LayerNorm(proj_dim),  # LN finale
        )

        # ── Tête ordinale ─────────────────────────────────────
        # K-1 seuils binaires indépendants (même logique que Cell 10)
        self.head = nn.Sequential(
            nn.Linear(proj_dim, proj_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim // 2, num_classes - 1),
        )

    def forward(self, img_emb, txt_emb, tab_emb):
        B = img_emb.size(0)

        # Projection [B, dim] → [B, 1, proj_dim]
        img = self.img_proj(img_emb).unsqueeze(1)
        txt = self.txt_proj(txt_emb).unsqueeze(1)
        tab = self.tab_proj(tab_emb).unsqueeze(1)

        # Token CLS répété sur le batch [1, 1, D] → [B, 1, D]
        cls = self.cls_token.expand(B, -1, -1)

        # Séquence [B, 4, proj_dim] : [CLS | img | txt | tab]
        seq = torch.cat([cls, img, txt, tab], dim=1)

        # Encodage positionnel additif
        seq = seq + self.pos_embed

        # Attention croisée entre toutes les modalités
        out = self.transformer(seq)   # [B, 4, proj_dim]

        # Classification depuis le token CLS (position 0)
        logits = self.head(out[:, 0, :])   # [B, num_classes-1]

        # Gates approximatifs pour l'affichage (norme des tokens en sortie)
        # Interprétation : modalité avec norme élevée → plus influente
        with torch.no_grad():
            norms = out[:, 1:, :].norm(dim=-1)          # [B, 3]
            gates = torch.softmax(norms, dim=-1)         # [B, 3]

        return logits, gates


print('CrossModalTransformer défini.')

CrossModalTransformer défini.


In [18]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 19 — RÉINSTANCIATION : fusion → CrossModalTransformer  ║
# ║                                                              ║
# ║  On réutilise model_img, model_txt, tab_encoder de Cell 11.  ║
# ║  On recharge aussi leurs meilleurs poids (best_model_state)  ║
# ║  sauvegardés en Cell 13 pour repartir du meilleur point.     ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Recharge les meilleurs poids des encodeurs ────────────────
# Ils ont déjà été entraînés par Cell 13 — on repart de ce point,
# pas de zéro. Seul `fusion` est remplacé par une architecture neuve.
if best_model_state is not None:
    model_img.load_state_dict(best_model_state['model_img'])
    model_txt.load_state_dict(best_model_state['model_txt'])
    tab_encoder.load_state_dict(best_model_state['tab_encoder'])
    print(f'Encodeurs rechargés depuis best checkpoint (val QWK = {best_val_qwk:.4f})')
else:
    print('Aucun checkpoint disponible — encodeurs initialisés aléatoirement.')

# ── Nouveau fusion : CrossModalTransformer ────────────────────
fusion_cmt = CrossModalTransformer(
    img_dim     = IMG_DIM,       # 1536
    txt_dim     = TXT_DIM,       # 768
    tab_dim     = TAB_ENC_DIM,   # 256
    num_classes = NUM_CLASSES,   # 5
    proj_dim    = 128,           # réduit vs 256 pour limiter l'overfit
    nhead       = 4,
    num_layers  = 2,
    dropout     = 0.15,
).float().to(device)

# Compte les paramètres
n_params = sum(p.numel() for p in fusion_cmt.parameters() if p.requires_grad)
print(f'CrossModalTransformer : {n_params/1e6:.2f}M paramètres')
print(f'proj_dim=128, nhead=4, num_layers=2')

Encodeurs rechargés depuis best checkpoint (val QWK = 0.3137)
CrossModalTransformer : 0.74M paramètres
proj_dim=128, nhead=4, num_layers=2


In [19]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 20 — OPTIMIZER & SCHEDULER POUR CMT                    ║
# ║                                                              ║
# ║  Stratégie : LR très bas pour les encodeurs (déjà entraînés) ║
# ║  LR normal pour fusion_cmt (from scratch)                    ║
# ╚══════════════════════════════════════════════════════════════╝
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS_CMT = 20   # plus d'epochs car encodeurs quasi-gelés → convergence lente

optimizer_cmt = torch.optim.AdamW([
    # Encodeurs : LR très faible — ils sont déjà bons, on les affine juste
    {'params': model_img.parameters(),   'lr': 5e-6},
    {'params': model_txt.parameters(),   'lr': 2e-6},
    {'params': tab_encoder.parameters(), 'lr': 2e-5},
    # CMT : LR normal — il part de zéro
    {'params': fusion_cmt.parameters(),  'lr': 5e-4},
], weight_decay=1e-2)

scheduler_cmt = CosineAnnealingLR(optimizer_cmt, T_max=EPOCHS_CMT, eta_min=1e-7)

print(f'Optimizer CMT prêt — {EPOCHS_CMT} epochs max')

Optimizer CMT prêt — 20 epochs max


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 21 — TRAINING LOOP CMT                                 ║
# ║                                                              ║
# ║  Identique à Cell 13 mais utilise fusion_cmt et             ║
# ║  optimizer_cmt / scheduler_cmt                               ║
# ║  evaluate() de Cell 13 est réutilisée telle quelle APRÈS    ║
# ║  avoir remplacé fusion par fusion_cmt (voir bas de cell)     ║
# ╚══════════════════════════════════════════════════════════════╝
import copy

# Swap temporaire : evaluate() utilise la variable `fusion`
# On pointe fusion sur fusion_cmt le temps de l'entraînement CMT
fusion_original = fusion   # sauvegarde l'ancien
fusion = fusion_cmt        # evaluate() utilisera fusion_cmt

best_val_qwk_cmt     = -1.0
patience_counter_cmt = 0
best_model_state_cmt = None
PATIENCE_CMT         = 6   # un peu plus généreux car le CMT converge plus lentement

scaler_amp_cmt = GradScaler()

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train QWK':>9} | "
      f"{'Val Loss':>8} | {'Val QWK':>7} | Gates (img/txt/tab)")
print('-' * 85)

for epoch in range(EPOCHS_CMT):

    model_img.train(); model_txt.train()
    tab_encoder.train(); fusion_cmt.train()

    total_loss, total_samples = 0.0, 0
    nan_batches = 0
    train_preds, train_labels = [], []
    gates = None

    optimizer_cmt.zero_grad()

    for step, (tab_b, txt_b, img_b, lbl_b) in enumerate(train_loader):
        tab_b = tab_b.float().to(device)
        img_b = img_b.float().to(device)
        lbl_b = lbl_b.to(device)

        with torch.cuda.amp.autocast():
            img_emb = model_img(img_b).float()
        txt_emb = get_txt_embedding(txt_b)
        tab_emb = tab_encoder(tab_b)

        if any(torch.isnan(e).any() for e in [img_emb, txt_emb, tab_emb]):
            nan_batches += 1
            continue

        logits, gates = fusion_cmt(img_emb, txt_emb, tab_emb)
        loss = ordinal_loss(logits, lbl_b, num_classes=NUM_CLASSES) / GRAD_ACCUM_STEPS

        if torch.isnan(loss):
            nan_batches += 1
            continue

        scaler_amp_cmt.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler_amp_cmt.unscale_(optimizer_cmt)
            torch.nn.utils.clip_grad_norm_(
                list(model_img.parameters()) + list(model_txt.parameters()) +
                list(tab_encoder.parameters()) + list(fusion_cmt.parameters()),
                max_norm=1.0
            )
            scaler_amp_cmt.step(optimizer_cmt)
            scaler_amp_cmt.update()
            optimizer_cmt.zero_grad()

        preds = ordinal_predict(logits)
        train_preds.extend(preds.cpu().numpy())
        train_labels.extend(lbl_b.cpu().numpy())
        total_loss    += loss.item() * GRAD_ACCUM_STEPS * len(lbl_b)
        total_samples += len(lbl_b)

    scheduler_cmt.step()

    if total_samples == 0:
        print(f'Epoch {epoch+1:02d} — tous les batches NaN.')
        continue

    train_loss = total_loss / total_samples
    train_qwk  = cohen_kappa_score(train_labels, train_preds, weights='quadratic')

    torch.cuda.empty_cache()
    val_qwk, val_loss = evaluate(val_loader)   # réutilise evaluate() de Cell 13
    torch.cuda.empty_cache()

    gates_str = 'gates=N/A'
    if gates is not None:
        g = gates.detach().float().mean(dim=0)
        gates_str = f'img={g[0]:.2f} txt={g[1]:.2f} tab={g[2]:.2f}'

    print(f'{epoch+1:>5d} | {train_loss:>10.4f} | {train_qwk:>9.4f} | '
          f'{val_loss:>8.4f} | {val_qwk:>7.4f} | {gates_str}'
          + (f' | NaN:{nan_batches}' if nan_batches else ''))

    if val_qwk > best_val_qwk_cmt:
        best_val_qwk_cmt     = val_qwk
        patience_counter_cmt = 0
        best_model_state_cmt = {
            'fusion_cmt':  copy.deepcopy(fusion_cmt.state_dict()),
            'tab_encoder': copy.deepcopy(tab_encoder.state_dict()),
            'model_img':   copy.deepcopy(model_img.state_dict()),
            'model_txt':   copy.deepcopy(model_txt.state_dict()),
        }
    else:
        patience_counter_cmt += 1
        if patience_counter_cmt >= PATIENCE_CMT:
            print(f'\n⚡ Early stopping epoch {epoch+1} '
                  f'(patience={PATIENCE_CMT}, best val QWK={best_val_qwk_cmt:.4f})')
            break

# ── Recharge le meilleur checkpoint CMT ──────────────────────
if best_model_state_cmt is not None:
    fusion_cmt.load_state_dict(best_model_state_cmt['fusion_cmt'])

Epoch | Train Loss | Train QWK | Val Loss | Val QWK | Gates (img/txt/tab)
-------------------------------------------------------------------------------------
    1 |     0.2939 |    0.7442 |   0.6346 |  0.2910 | img=0.34 txt=0.33 tab=0.33
    2 |     0.2437 |    0.8113 |   0.6735 |  0.3166 | img=0.33 txt=0.33 tab=0.33
    3 |     0.2231 |    0.8333 |   0.7225 |  0.3168 | img=0.33 txt=0.33 tab=0.34


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 22 — COMPARAISON OrdinalFusion vs CMT                  ║
# ║  Choisit automatiquement le meilleur pour la soumission      ║
# ╚══════════════════════════════════════════════════════════════╝

print('═' * 55)
print(f'  OrdinalFusionClassifier  val QWK : {best_val_qwk:.4f}')
print(f'  CrossModalTransformer    val QWK : {best_val_qwk_cmt:.4f}')
print('═' * 55)

if best_val_qwk_cmt > best_val_qwk:
    print('\n→ CMT gagne — on utilise fusion_cmt pour la soumission')
    fusion = fusion_cmt

    # Recharge les poids CMT dans les encodeurs
    if best_model_state_cmt:
        fusion_cmt.load_state_dict(best_model_state_cmt['fusion_cmt'])
        tab_encoder.load_state_dict(best_model_state_cmt['tab_encoder'])
        model_img.load_state_dict(best_model_state_cmt['model_img'])
        model_txt.load_state_dict(best_model_state_cmt['model_txt'])
    BEST_MODEL_NAME = 'CrossModalTransformer'
    BEST_QWK        = best_val_qwk_cmt

else:
    print('\n→ OrdinalFusion gagne — on garde fusion original')
    fusion = fusion_original

    if best_model_state:
        fusion.load_state_dict(best_model_state['fusion'])
        tab_encoder.load_state_dict(best_model_state['tab_encoder'])
        model_img.load_state_dict(best_model_state['model_img'])
        model_txt.load_state_dict(best_model_state['model_txt'])
    BEST_MODEL_NAME = 'OrdinalFusionClassifier'
    BEST_QWK        = best_val_qwk

print(f'\nModèle retenu : {BEST_MODEL_NAME} (val QWK = {BEST_QWK:.4f})')
print('Les cells 14 (LGBM) et 15 (Submission) utilisent maintenant ce modèle.')